# Загальна таблиця на рівні клієнта

Рядок = клієнт: сегмент, портфель ТЗ, вартість, вік, найдорожче авто, прапорці `HAS_*`,
залишки, доходи, схильності, активність у Finalta.

Самостійний ноутбук: використовує лише базові функції `utils.py` (підготовка й агрегація ТЗ), нічого в інших файлах міняти не треба.
Запускати після `pipeline.ipynb`.

Результат:
- `output/client_table.xlsx` — для бізнесу (українські назви колонок, форматування, аркуш з описом колонок);
- `output/client_table.parquet` — для Python / SQL (технічні назви колонок).

In [ ]:
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

import utils
importlib.reload(utils)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 220)

## 0. Налаштування

In [ ]:
DATA_PATH = Path(r'C:\repo\data-science\Nazar Hnatyshyn\RETAIL\(DS-1052) Vehicle market analysis\df_for_clustering.xlsx')
SHEET_NAME = 0
OUTPUT_DIR = Path("output")
EXCEL_PATH = OUTPUT_DIR / "client_table.xlsx"
PARQUET_PATH = OUTPUT_DIR / "client_table.parquet"

MONTH_DATA = "2026_10"      # залишки, доходи, Finalta
MONTH_SCORING = "2026_11"   # скоринги депозитів і ОВДП

BASE = Path(r"M:\Controlling\Data_Science_Projects")
LIABILITIES_PATH = BASE / "Common_Data" / "Processed" / f"dataset_{MONTH_DATA}.parquet"
INCOME_PATH = BASE / "Common_Data" / "Raw" / f"info_dataset_{MONTH_DATA}.parquet"
DEPOSIT_PATH = BASE / "Scorings" / MONTH_SCORING / f"deposits_propensity_{MONTH_SCORING}.parquet"
OVDP_PATH = BASE / "Scorings" / MONTH_SCORING / f"dgb_{MONTH_SCORING}.parquet"
FINALTA_PATH = BASE / "Activity" / f"finalta_{MONTH_DATA}.csv"

INCOME_TYPES = {
    "SAL": "Зарплата", "SOC": "Соціальні виплати", "INT": "Відсотки", "BNS": "Бонуси",
    "RNT": "Оренда", "DIV": "Дивіденди", "ROY": "Роялті",
}

for path in [DATA_PATH, OUTPUT_DIR / "customer_segments.csv.gz", OUTPUT_DIR / "segmentation_config.json",
             LIABILITIES_PATH, INCOME_PATH, DEPOSIT_PATH, OVDP_PATH, FINALTA_PATH]:
    print("OK     " if path.exists() else "НЕМАЄ  ", path)

## 1. Портфель ТЗ

In [ ]:
config = utils.load_json(OUTPUT_DIR / "segmentation_config.json")
raw = utils.load_raw(DATA_PATH, SHEET_NAME)
vehicles = utils.prepare_vehicle_data(raw)
customers = utils.aggregate_customers(vehicles, age_median=config["age_median"])
print(f"ТЗ: {len(vehicles):,} | клієнтів: {len(customers):,}")

v = vehicles
passenger = v["IS_PASSENGER"].eq(1)

# Вартість і вік
value_all = v.groupby("user_id")["MARKET_VALUE_USD"].agg(ALL_VALUE_MEAN="mean", ALL_VALUE_MIN="min",
                                                         ALL_VALUE_MAX="max", ALL_VALUE_SUM="sum")
value_cars = v.loc[passenger].groupby("user_id")["MARKET_VALUE_USD"].agg(CAR_VALUE_MEAN="mean", CAR_VALUE_MIN="min",
                                                                         CAR_VALUE_MAX="max", CAR_VALUE_SUM="sum")
age_cars = v.loc[passenger].groupby("user_id")["VEHICLE_AGE_CLEAN"].agg(CAR_AGE_MEAN="mean", CAR_AGE_MIN="min",
                                                                        CAR_AGE_MAX="max")

# Кількість легкових за ціновими рівнями
tiers = (pd.crosstab(v.loc[passenger, "user_id"], v.loc[passenger, "VALUE_TIER"])
           .reindex(columns=["T1", "T2", "T3", "T4", "T5"], fill_value=0)
           .add_prefix("CARS_"))

# Найдорожче авто: серед легкових, а якщо легкових немає — серед усіх ТЗ
class_names = {"PASSENGER": "Легкове", "COMMERCIAL": "Вантажне / спецтехніка", "BUS": "Автобус",
               "MOTO": "Мото", "TRAILER": "Причіп", "OTHER": "Інше"}
ranked = v.assign(_CAR=passenger.astype(int), _VAL=v["MARKET_VALUE_USD"].fillna(-1))
ranked = ranked.sort_values(["user_id", "_CAR", "_VAL"], ascending=[True, False, False])
top = ranked.drop_duplicates("user_id").set_index("user_id")
top = pd.DataFrame({
    "TOP_VEHICLE_CLASS": top["VEHICLE_CLASS"].map(class_names),
    "TOP_VEHICLE_BRAND": top["BRAND_STD"],
    "TOP_VEHICLE_MODEL": top["MODEL_STD"],
    "TOP_VEHICLE_YEAR": top["PRODUCTION_YEAR_CLEAN"],
    "TOP_VEHICLE_VALUE_USD": top["MARKET_VALUE_USD"],
})
display(top.head())

## 2. Сегменти

In [ ]:
segments = pd.read_csv(OUTPUT_DIR / "customer_segments.csv.gz")
keep = [c for c in ["user_id", "FINAL_SEGMENT", "SEGMENT_SOURCE", "KMEANS_CLUSTER"] if c in segments.columns]
segments = segments[keep]

SEGMENT_DISPLAY = {
    "LUXURY": "Luxury Vehicle Owners", "MODERN_PREMIUM": "Modern Premium Owners",
    "ESTABLISHED_PREMIUM": "Established Premium Owners", "ESTABLISHED_PREMIUM_SUV": "Established Premium SUV Owners",
    "MULTI_CAR_3PLUS": "Multi-Car Owners (3+)", "MULTI_VEHICLE_HOUSEHOLDS": "Multi-Vehicle Households",
    "MAINSTREAM_SUV": "Mainstream SUV Owners", "EV_ADOPTERS": "Electric Vehicle Adopters",
    "CORE_MASS_MARKET_LOWER_MID": "Core Mass Market Lower-Mid", "CORE_MASS_MARKET_BUDGET": "Core Mass Market Budget",
    "BUDGET_LEGACY": "Budget Legacy Owners", "BUSINESS_VEHICLE_OWNERS": "Business Vehicle Owners",
    "COMMERCIAL_VEHICLE_OWNERS": "Commercial Vehicle Owners", "MOTORCYCLE_OWNERS": "Motorcycle Owners",
    "TRAILER_ONLY_OWNERS": "Trailer-Only Owners", "OTHER_NON_PASSENGER": "Other (без легкового авто)",
}
SEGMENT_GROUPS = {
    "Заможні": ["LUXURY", "MODERN_PREMIUM", "ESTABLISHED_PREMIUM_SUV", "MULTI_CAR_3PLUS"],
    "Established Premium": ["ESTABLISHED_PREMIUM"],
    "Масові": ["CORE_MASS_MARKET_LOWER_MID", "MAINSTREAM_SUV", "MULTI_VEHICLE_HOUSEHOLDS", "EV_ADOPTERS"],
    "Бюджетні": ["CORE_MASS_MARKET_BUDGET", "BUDGET_LEGACY"],
    "Бізнес": ["BUSINESS_VEHICLE_OWNERS", "COMMERCIAL_VEHICLE_OWNERS"],
    "Без легкового авто": ["MOTORCYCLE_OWNERS", "TRAILER_ONLY_OWNERS", "OTHER_NON_PASSENGER"],
}
group_of = {seg: grp for grp, segs in SEGMENT_GROUPS.items() for seg in segs}
segments["SEGMENT_NAME"] = segments["FINAL_SEGMENT"].map(lambda s: SEGMENT_DISPLAY.get(s, s))
segments["SEGMENT_GROUP"] = segments["FINAL_SEGMENT"].map(lambda s: group_of.get(s, "Інші"))
print(f"Клієнтів у сегментації: {len(segments):,}")

## 3. Залишки, доходи, схильності, Finalta

In [ ]:
def load_keyed(df, key, how):
    """Приводить ключ до user_id (int64) і прибирає дублікати."""
    df = df.rename(columns={key: "user_id"})
    df["user_id"] = df["user_id"].astype("int64")
    dup = int(df["user_id"].duplicated().sum())
    if dup:
        print(f"  дублікатів ключа: {dup:,} → {how}")
        df = df.groupby("user_id", as_index=False).agg(how)
    return df


liab = load_keyed(pd.read_parquet(LIABILITIES_PATH, columns=["CONTRAGENTID", "LIABILITIES"]), "CONTRAGENTID", "sum")
liab["LIABILITIES"] = liab["LIABILITIES"].astype("float64")

income_cols = {f"{code}_{MONTH_DATA}": f"INCOME_{code}" for code in INCOME_TYPES}
inc = pd.read_parquet(INCOME_PATH, columns=["CONTRAGENTID"] + list(income_cols)).rename(columns=income_cols)
inc = load_keyed(inc, "CONTRAGENTID", "sum")
inc["TOTAL_INCOME"] = inc[list(income_cols.values())].fillna(0).sum(axis=1)

dep = pd.read_parquet(DEPOSIT_PATH, columns=["CONTRAGENTID", "PROB_TO_BUY"]).rename(columns={"PROB_TO_BUY": "PROB_TO_DEPOSIT"})
dep = load_keyed(dep, "CONTRAGENTID", "first")
ovdp = pd.read_parquet(OVDP_PATH, columns=["CONTRAGENTID", "PROB_TO_BUY"]).rename(columns={"PROB_TO_BUY": "PROB_TO_OVDP"})
ovdp = load_keyed(ovdp, "CONTRAGENTID", "first")
active_ids = set(pd.read_csv(FINALTA_PATH, usecols=["CLIENT_ID"])["CLIENT_ID"].astype("int64"))
print("Джерела завантажено")

## 4. Збираємо таблицю

In [ ]:
c = customers.copy()
total = c["TOTAL_VEHICLE_COUNT"]
c["PASSENGER_SHARE_ALL"] = c["PASSENGER_VEHICLE_COUNT"] / total
c["COMMERCIAL_SHARE"] = c["COMMERCIAL_COUNT"] / total
c["BUS_SHARE"] = c["BUS_COUNT"] / total
c["MOTO_SHARE"] = c["MOTO_COUNT"] / total
# TRAILER_SHARE уже є в агрегації

table = (segments
         .merge(c, on="user_id", how="left")
         .merge(value_cars, left_on="user_id", right_index=True, how="left")
         .merge(value_all, left_on="user_id", right_index=True, how="left")
         .merge(age_cars, left_on="user_id", right_index=True, how="left")
         .merge(tiers, left_on="user_id", right_index=True, how="left")
         .merge(top, left_on="user_id", right_index=True, how="left")
         .merge(liab, on="user_id", how="left")
         .merge(inc, on="user_id", how="left")
         .merge(dep, on="user_id", how="left")
         .merge(ovdp, on="user_id", how="left"))
table[[f"CARS_T{i}" for i in range(1, 6)]] = table[[f"CARS_T{i}" for i in range(1, 6)]].fillna(0).astype(int)
table["IS_ACTIVE_FINALTA"] = table["user_id"].isin(active_ids).astype(int)
table["IN_LIABILITIES_FILE"] = table["LIABILITIES"].notna().astype(int)
table["IN_INCOME_FILE"] = table["TOTAL_INCOME"].notna().astype(int)
table["IN_DEPOSIT_SCORING"] = table["PROB_TO_DEPOSIT"].notna().astype(int)
table["IN_OVDP_SCORING"] = table["PROB_TO_OVDP"].notna().astype(int)
assert len(table) == len(segments) and table["user_id"].is_unique
print(f"Клієнтів: {len(table):,}")

In [ ]:
# Колонки у фінальному порядку: (технічна назва, назва в Excel, формат)
F_INT, F_USD, F_PCT, F_YEAR, F_AGE, F_PROB, F_TXT = "int", "usd", "pct", "year", "age", "prob", "txt"
COLUMNS = [
    # Сегмент
    ("user_id", "ID клієнта", F_TXT),
    ("FINAL_SEGMENT", "Код сегмента", F_TXT),
    ("SEGMENT_NAME", "Сегмент", F_TXT),
    ("SEGMENT_GROUP", "Група сегмента", F_TXT),
    ("SEGMENT_SOURCE", "Як присвоєно (RULE / TREE)", F_TXT),
    ("KMEANS_CLUSTER", "Кластер KMeans", F_INT),
    # Кількість ТЗ
    ("TOTAL_VEHICLE_COUNT", "ТЗ усього", F_INT),
    ("PASSENGER_VEHICLE_COUNT", "Легкових", F_INT),
    ("COMMERCIAL_COUNT", "Вантажних / спецтехніки", F_INT),
    ("BUS_COUNT", "Автобусів", F_INT),
    ("MOTO_COUNT", "Мото", F_INT),
    ("TRAILER_COUNT", "Причепів усього", F_INT),
    ("LIGHT_TRAILER_COUNT", "Причепів легких", F_INT),
    ("HEAVY_TRAILER_COUNT", "Причепів важких", F_INT),
    ("BOAT_TRAILER_COUNT", "Причепів для човна", F_INT),
    ("CAMPER_TRAILER_COUNT", "Причепів житлових", F_INT),
    ("BUSINESS_VEHICLE_COUNT", "Бізнес-ТЗ (вантажні + автобуси + важкі причепи)", F_INT),
    # Частки від усіх ТЗ
    ("PASSENGER_SHARE_ALL", "Частка легкових", F_PCT),
    ("COMMERCIAL_SHARE", "Частка вантажних / спецтехніки", F_PCT),
    ("BUS_SHARE", "Частка автобусів", F_PCT),
    ("MOTO_SHARE", "Частка мото", F_PCT),
    ("TRAILER_SHARE", "Частка причепів", F_PCT),
    # Вартість
    ("CAR_VALUE_MEAN", "Легкові: середня вартість, USD", F_USD),
    ("CAR_VALUE_MIN", "Легкові: мін. вартість, USD", F_USD),
    ("CAR_VALUE_MAX", "Легкові: макс. вартість, USD", F_USD),
    ("CAR_VALUE_SUM", "Легкові: сумарна вартість, USD", F_USD),
    ("ALL_VALUE_MEAN", "Усі ТЗ: середня вартість, USD", F_USD),
    ("ALL_VALUE_MIN", "Усі ТЗ: мін. вартість, USD", F_USD),
    ("ALL_VALUE_MAX", "Усі ТЗ: макс. вартість, USD", F_USD),
    ("ALL_VALUE_SUM", "Усі ТЗ: сумарна вартість, USD", F_USD),
    # Вік легкових
    ("CAR_AGE_MEAN", "Легкові: середній вік, років", F_AGE),
    ("CAR_AGE_MIN", "Легкові: вік наймолодшого, років", F_AGE),
    ("CAR_AGE_MAX", "Легкові: вік найстаршого, років", F_AGE),
    # Найдорожче авто
    ("TOP_VEHICLE_CLASS", "Найдорожче ТЗ: тип", F_TXT),
    ("TOP_VEHICLE_BRAND", "Найдорожче ТЗ: марка", F_TXT),
    ("TOP_VEHICLE_MODEL", "Найдорожче ТЗ: модель", F_TXT),
    ("TOP_VEHICLE_YEAR", "Найдорожче ТЗ: рік", F_YEAR),
    ("TOP_VEHICLE_VALUE_USD", "Найдорожче ТЗ: вартість, USD", F_USD),
    # Склад легкових
    ("PREMIUM_COUNT", "Premium-авто", F_INT),
    ("PREMIUM_SHARE", "Частка premium серед легкових", F_PCT),
    ("SUV_COUNT", "SUV", F_INT),
    ("SUV_SHARE", "Частка SUV серед легкових", F_PCT),
    ("EV_COUNT", "Електромобілів", F_INT),
    ("EV_SHARE", "Частка EV серед легкових", F_PCT),
    ("HYBRID_COUNT", "Hybrid", F_INT),
    ("HYBRID_SHARE", "Частка hybrid серед легкових", F_PCT),
    ("GAS_COUNT", "На газі", F_INT),
    ("GAS_SHARE", "Частка газ серед легкових", F_PCT),
    ("CARS_T1", "Легкових до $4k", F_INT),
    ("CARS_T2", "Легкових $4–14k", F_INT),
    ("CARS_T3", "Легкових $14–30k", F_INT),
    ("CARS_T4", "Легкових $30–60k", F_INT),
    ("CARS_T5", "Легкових від $60k", F_INT),
    # Прапорці
    ("HAS_PREMIUM", "Є premium", F_INT),
    ("HAS_SUV", "Є SUV", F_INT),
    ("HAS_EV", "Є EV", F_INT),
    ("HAS_HYBRID", "Є hybrid", F_INT),
    ("HAS_GAS", "Є авто на газі", F_INT),
    ("HAS_LUXURY_VEHICLE", "Є авто від $60k", F_INT),
    ("HAS_MODERN_PREMIUM", "Є modern premium", F_INT),
    ("HAS_TRAILER", "Є причіп", F_INT),
    ("HAS_LIGHT_TRAILER", "Є легкий причіп", F_INT),
    ("HAS_HEAVY_TRAILER", "Є важкий причіп", F_INT),
    ("HAS_BOAT_TRAILER", "Є причіп для човна", F_INT),
    ("HAS_CAMPER_TRAILER", "Є житловий причіп", F_INT),
    ("HAS_COMMERCIAL", "Є вантажне / спецтехніка", F_INT),
    ("HAS_BUS", "Є автобус", F_INT),
    ("HAS_MOTO", "Є мото", F_INT),
    # Фінанси
    ("IS_ACTIVE_FINALTA", "Активний (Finalta)", F_INT),
    ("IN_LIABILITIES_FILE", "Є дані про залишки", F_INT),
    ("LIABILITIES", "Залишки, UAH", F_USD),
    ("IN_INCOME_FILE", "Є дані про доходи", F_INT),
    ("TOTAL_INCOME", "Total income, UAH", F_USD),
] + [(f"INCOME_{code}", f"{name}, UAH", F_USD) for code, name in INCOME_TYPES.items()] + [
    ("IN_DEPOSIT_SCORING", "Є скор депозиту", F_INT),
    ("PROB_TO_DEPOSIT", "Схильність до депозиту", F_PROB),
    ("IN_OVDP_SCORING", "Є скор ОВДП", F_INT),
    ("PROB_TO_OVDP", "Схильність до ОВДП", F_PROB),
]
COLUMNS = [col for col in COLUMNS if col[0] in table.columns]
final = table[[col[0] for col in COLUMNS]].copy()

# Для клієнтів без легкових вік і частки легкових не мають сенсу
no_cars = final["PASSENGER_VEHICLE_COUNT"].eq(0)
for col in ["PREMIUM_SHARE", "SUV_SHARE", "EV_SHARE", "HYBRID_SHARE", "GAS_SHARE"]:
    final.loc[no_cars, col] = np.nan

print(f"Колонок: {final.shape[1]}")
display(final.head(10))

## 5. Збереження

In [ ]:
final.to_parquet(PARQUET_PATH, index=False)
print("Збережено:", PARQUET_PATH.resolve())

FORMATS = {"int": "#,##0", "usd": "#,##0", "pct": "0.0%", "year": "0", "age": "0.0", "prob": "0.0000", "txt": "@"}
WIDTHS = {"int": 11, "usd": 14, "pct": 11, "year": 9, "age": 11, "prob": 13, "txt": 18}
headers = [col[1] for col in COLUMNS]
kinds = [col[2] for col in COLUMNS]
values = final.astype(object).where(final.notna(), None).values.tolist()

try:
    import xlsxwriter
    wb = xlsxwriter.Workbook(EXCEL_PATH, {"constant_memory": True})
    ws = wb.add_worksheet("Клієнти")
    head_fmt = wb.add_format({"bold": True, "font_color": "white", "bg_color": "#1F3864", "text_wrap": True,
                              "align": "center", "valign": "vcenter", "border": 1})
    col_fmts = {k: wb.add_format({"num_format": f}) for k, f in FORMATS.items()}
    for j, kind in enumerate(kinds):
        ws.set_column(j, j, 30 if headers[j] == "Сегмент" else WIDTHS[kind], col_fmts[kind])
    ws.set_row(0, 48)
    ws.write_row(0, 0, headers, head_fmt)
    for i, row in enumerate(values, start=1):
        for j, value in enumerate(row):
            if value is not None:
                ws.write(i, j, value, col_fmts[kinds[j]])
    ws.freeze_panes(1, 3)
    ws.autofilter(0, 0, len(values), len(headers) - 1)

    ws2 = wb.add_worksheet("Опис колонок")
    ws2.set_column(0, 0, 45)
    ws2.set_column(1, 1, 30)
    ws2.write_row(0, 0, ["Колонка", "Технічна назва"], head_fmt)
    for i, (code, name, _) in enumerate(COLUMNS, start=1):
        ws2.write_row(i, 0, [name, code])
    wb.close()
except ImportError:
    # Запасний варіант без xlsxwriter: openpyxl (без числових форматів колонок)
    from openpyxl import Workbook
    from openpyxl.styles import Font, PatternFill
    wb = Workbook(write_only=True)
    ws = wb.create_sheet("Клієнти")
    ws.freeze_panes = "D2"
    ws.append(headers)
    for row in values:
        ws.append(row)
    ws2 = wb.create_sheet("Опис колонок")
    ws2.append(["Колонка", "Технічна назва"])
    for code, name, _ in COLUMNS:
        ws2.append([name, code])
    wb.save(EXCEL_PATH)
    print("xlsxwriter не встановлено — збережено без форматування (pip install xlsxwriter)")

print("Збережено:", EXCEL_PATH.resolve(), f"| {len(final):,} рядків × {final.shape[1]} колонок")